# Notebook 02 — Feature Engineering

**Goal:** Compute all behavioural features from the enriched scrobble data and produce a user-level feature matrix for clustering.

**Inputs** (from `data/processed/`):
- `scrobbles_updated.parquet`
- `profiles.parquet`
- `artist_genres.parquet`
- `audio_features.parquet`

**Outputs:**
- `data/processed/user_features.parquet` — one row per user, all features

**Features computed:**

| Group | Features |
|---|---|
| Artist diversity | unique_artists, artist_entropy, artist_concentration_20 |
| Genre diversity | unique_genres, genre_entropy, genre_concentration_5, avg_genre_tags_per_play |
| Engagement | total_scrobbles, unique_tracks, track_replay_rate, avg_tracks_per_session, session_count |
| Discovery | discovery_velocity_30d/90d, novelty_ratio, top_artist_play_share |
| Temporal | temporal_hour/dow/month_entropy, morning/evening/weekend_ratio, temporal_stability_pc1..5 |
| Audio profile | mean danceability, energy, valence, tempo, acousticness, instrumentalness |

In [ ]:
import sys
import logging
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# ── Memory-efficient loading for large scrobbles_updated.parquet ────────────
#
# Strategy:
#   1. Read only the 'userid' column to discover all users (very cheap).
#   2. Load auxiliary files fully — they are small (< a few MB each).
#   3. Feature cells stream through the Parquet in user-batches, loading
#      only the columns each feature module actually needs.
#
# Tune USERS_PER_BATCH to your available RAM:
#   • Lower  → less peak RAM, more passes over the file
#   • Higher → faster, but uses more RAM per batch
#   Typical starting point: 50–150 users per batch.
# ────────────────────────────────────────────────────────────────────────────

import pandas as pd
from src.data.loader import (
    get_parquet_userids,
    load_parquet_chunked,
    stream_scrobbles_by_user_batch,
)

SCROBBLES_PATH   = '../data/processed/scrobbles_updated.parquet'
USERS_PER_BATCH  = 100  # ← adjust based on your RAM

# Cheaply get all unique user IDs (reads only 1 column)
all_userids = get_parquet_userids(SCROBBLES_PATH)

# These auxiliary files are small — load them fully as before
profiles       = pd.read_parquet('../data/processed/profiles.parquet')
artist_genres  = pd.read_parquet('../data/processed/artist_genres.parquet')
audio_features = pd.read_parquet('../data/processed/audio_features.parquet')

print(f'Total users in scrobbles : {len(all_userids):,}')
print(f'User batch size          : {USERS_PER_BATCH}  →  '
      f'{(len(all_userids) + USERS_PER_BATCH - 1) // USERS_PER_BATCH} batches')
print(f'Artist genres            : {len(artist_genres):,} artists, '
      f'{artist_genres["spotify_artist_id"].notna().sum():,} matched')
print(f'Audio features           : {len(audio_features):,} tracks')

## 1. Artist Diversity Features

In [ ]:
from src.features.diversity import compute_artist_diversity

# Only 'userid' and 'artist_name' are needed — load 2 of the 6 columns.
_chunks = []
for chunk in stream_scrobbles_by_user_batch(
        SCROBBLES_PATH, all_userids,
        users_per_batch=USERS_PER_BATCH,
        columns=['userid', 'artist_name']):
    _chunks.append(compute_artist_diversity(chunk, top_n=20))

artist_div = pd.concat(_chunks)
del _chunks  # free intermediate memory

print(f'Artist diversity features: {artist_div.shape}')
artist_div.describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
artist_div['unique_artists'].hist(ax=axes[0], bins=30)
axes[0].set_title('Unique Artists per User')

artist_div['artist_entropy'].hist(ax=axes[1], bins=30)
axes[1].set_title('Artist Shannon Entropy')

artist_div['artist_concentration_20'].hist(ax=axes[2], bins=30)
axes[2].set_title('Top-20 Artist Concentration')

plt.tight_layout()
plt.show()

## 2. Genre Diversity Features

In [ ]:
from src.features.diversity import compute_genre_diversity

# Only 'userid' and 'artist_name' are needed for genre lookup.
_chunks = []
for chunk in stream_scrobbles_by_user_batch(
        SCROBBLES_PATH, all_userids,
        users_per_batch=USERS_PER_BATCH,
        columns=['userid', 'artist_name']):
    _chunks.append(compute_genre_diversity(chunk, artist_genres, top_n=5))

genre_div = pd.concat(_chunks)
del _chunks

print(f'Genre diversity features: {genre_div.shape}')
genre_div.describe()

## 3. Temporal Features

Includes PCA-compressed hourly listening profiles to capture **listening pattern stability** while avoiding multicollinearity across 24 hour-of-day columns.

In [ ]:
from src.features.temporal import compute_temporal_features

# Temporal features need 'userid' + 'timestamp' only — 2 of 6 columns.
# Loading just these 2 columns for all users is small enough to fit in RAM
# (timestamps are compact), and it is REQUIRED to load all users at once
# so that the PCA is fitted globally (consistent axes across all users).
print('Loading userid + timestamp (2 columns only)...')
scrobbles_temporal = pd.concat(
    list(load_parquet_chunked(SCROBBLES_PATH, columns=['userid', 'timestamp']))
)
print(f'  Loaded {len(scrobbles_temporal):,} rows, {scrobbles_temporal["userid"].nunique()} users')

temporal = compute_temporal_features(scrobbles_temporal, pca_components=5)
del scrobbles_temporal  # free memory — PCA is already fitted

print(f'Temporal features: {temporal.shape}')
temporal.describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
temporal['temporal_hour_entropy'].hist(ax=axes[0], bins=20)
axes[0].set_title('Hour-of-Day Entropy')

temporal['weekend_ratio'].hist(ax=axes[1], bins=20)
axes[1].set_title('Weekend Listen Ratio')

temporal['avg_daily_plays'].hist(ax=axes[2], bins=30)
axes[2].set_title('Avg Daily Plays')

plt.tight_layout()
plt.show()

## 4. Engagement & Discovery Features

In [ ]:
from src.features.engagement import compute_engagement_features

# Engagement needs 4 columns.  User-batch so only USERS_PER_BATCH users'
# rows are in RAM at a time.  Session detection is per-user, so batching
# by users (not by row-groups) is correct — each user's full history is
# always in the same batch.
_chunks = []
for chunk in stream_scrobbles_by_user_batch(
        SCROBBLES_PATH, all_userids,
        users_per_batch=USERS_PER_BATCH,
        columns=['userid', 'timestamp', 'artist_name', 'track_name']):
    _chunks.append(compute_engagement_features(chunk, session_gap_minutes=30))

engagement = pd.concat(_chunks)
del _chunks

print(f'Engagement features: {engagement.shape}')
engagement.describe()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(['track_replay_rate', 'avg_tracks_per_session',
                          'discovery_velocity_30d', 'novelty_ratio',
                          'top_artist_play_share', 'session_count']):
    engagement[col].hist(ax=axes[i], bins=25)
    axes[i].set_title(col.replace('_', ' ').title())

plt.tight_layout()
plt.show()

## 5. Audio Feature Profile

In [ ]:
from src.features.engagement import compute_audio_feature_profile

# Audio profile needs 3 columns.
_chunks = []
for chunk in stream_scrobbles_by_user_batch(
        SCROBBLES_PATH, all_userids,
        users_per_batch=USERS_PER_BATCH,
        columns=['userid', 'artist_name', 'track_name']):
    _chunks.append(compute_audio_feature_profile(chunk, audio_features))

audio_profile = pd.concat(_chunks)
del _chunks

print(f'Audio feature profile: {audio_profile.shape}')
audio_profile.head()

## 6. Build Final Feature Matrix

In [ ]:
import os

# All feature blocks were already computed above in a memory-efficient way.
# Join them directly — no need to reload scrobbles or call build_feature_matrix.
feature_matrix = (
    artist_div
    .join(genre_div,    how='outer')
    .join(temporal,     how='outer')
    .join(engagement,   how='outer')
    .join(audio_profile, how='outer')
)

# Add demographics (descriptive, not used as clustering inputs)
profiles_indexed = profiles.set_index('userid')[['gender', 'age', 'country']]
feature_matrix = feature_matrix.join(profiles_indexed, how='left').sort_index()

os.makedirs('../data/processed', exist_ok=True)
feature_matrix.to_parquet('../data/processed/user_features.parquet')

print(f'Feature matrix: {feature_matrix.shape[0]} users × {feature_matrix.shape[1]} features')
feature_matrix.head()

## 7. Correlation Analysis & Multicollinearity Check

In [ ]:
# Select numeric clustering features (exclude demographics)
clustering_features = feature_matrix.select_dtypes(include=[np.number]).drop(
    columns=['age'], errors='ignore'
)

corr = clustering_features.corr()

plt.figure(figsize=(20, 16))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, cmap='RdBu', center=0, vmin=-1, vmax=1,
    annot=False, fmt='.2f', linewidths=0.3,
)
plt.title('Feature Correlation Matrix (Clustering Features)')
plt.tight_layout()
plt.savefig('../outputs/figures/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Pairs with |r| > 0.75
high_corr = (
    corr.abs().where(np.tril(np.ones(corr.shape), k=-1).astype(bool))
    .stack().reset_index()
    .rename(columns={0: 'correlation', 'level_0': 'feat_a', 'level_1': 'feat_b'})
    .query('correlation > 0.75')
    .sort_values('correlation', ascending=False)
)
print(f'Highly correlated pairs (|r|>0.75): {len(high_corr)}')
print(high_corr.to_string())

> Note: Remaining multicollinearity is handled at the clustering stage via PCA dimensionality reduction before fitting KMeans/HDBSCAN.

Proceed to **Notebook 03** for clustering.